In [ ]:
# Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load Video Index

video_index = pd.read_csv("../vid_index/video_index.csv")

print("Video index loaded ✅")
print("Shape:", video_index.shape)

video_index.head()

Video index loaded ✅
Shape: (245, 388)


,video_id,title,datetime,transcript,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,ldkfMvE36FI,How to handle being thrown into an existing co...,2026-03-08 12:43:26+00:00,theres going to be a need for people can like ...,-0.017596,-0.095291,0.056069,-0.028207,0.039805,-0.015040,...,0.079234,0.013098,-0.023241,0.036684,-0.066737,0.071216,-0.078861,-0.008572,0.032594,0.015356
1,jie_039IekA,Why you shouldnt chase people to get what you ...,2026-03-07 13:32:30+00:00,I dont do a ton of followup Like you know if i...,-0.058301,0.013225,0.012424,-0.075241,-0.012175,-0.032885,...,0.013678,-0.054440,0.027800,0.021932,0.046389,0.025415,0.005570,-0.002358,-0.005379,0.029290
2,mrRfPVm9nAY,Learn the basics of Data Structures in 60 seco...,2026-03-06 13:18:50+00:00,Lets learn the basics of data structures in le...,-0.015724,0.019981,0.005629,-0.015846,-0.079441,-0.097639,...,0.022740,0.014300,0.000673,-0.036183,0.020679,0.021774,0.002178,0.013053,-0.016505,-0.004650
3,hP931079TMw,There are 2 kinds of devs One of them is screw...,2026-03-06 11:01:15+00:00,Welcome back to the Free Code Camp podcast Im ...,-0.056384,-0.045460,0.002062,-0.044062,-0.019443,-0.011673,...,0.056522,0.045513,0.004705,-0.057094,-0.025583,0.024735,-0.003302,0.036114,0.018302,-0.008661
4,tVskbekONlw,Learn MLOps with MLflow and Databricks Full Co...,2026-03-05 14:21:18+00:00,This course is an end toend guide to mastering...,-0.022512,-0.077987,0.026442,-0.015346,0.055578,-0.076017,...,0.089358,0.048688,0.019932,-0.056607,0.027377,0.053911,0.003975,0.035524,-0.008475,0.013456


In [ ]:
#  Extract Embeddings
embedding_cols = [col for col in video_index.columns if "embedding_" in col]

embeddings = video_index[embedding_cols].values

print("Embeddings shape:", embeddings.shape)


Embeddings shape: (245, 384)


In [ ]:
# -----------------------------
#  Load Model
# -----------------------------

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print(f"Model Loaded: {model_name} ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6620.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded: all-MiniLM-L6-v2 ✅


In [ ]:
# -----------------------------
# Query Encoding + Scoring
# -----------------------------

query = "What is overfitting?"

# Encode query
query_embedding = model.encode([query])

# Compute similarity with stored embeddings
scores = cosine_similarity(query_embedding, embeddings)[0]

# Debug info
print("Max score:", max(scores))
print("Min score:", min(scores))

Max score: 0.31031464723097607
Min score: -0.10422099433944892


In [ ]:
# -----------------------------
# Rank Results
# -----------------------------

top_k = 5

top_indices = np.argsort(scores)[::-1][:top_k]

results = video_index.iloc[top_indices][['video_id', 'title']].copy()
results['score'] = scores[top_indices]

results

,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001
127,9kKNFP9srjc,So how do closures work again,0.261079
9,E7o_WdfKszU,Theres so much to learn so how do you focus to...,0.260392
30,oRZouQxC5Gw,When youre learning a new skill youve gotta be...,0.250224


In [ ]:
# -----------------------------
# Threshold Filtering
# -----------------------------

threshold = 0.3

filtered_indices = [i for i in top_indices if scores[i] > threshold]

# fallback if no results
if len(filtered_indices) == 0:
    print("⚠️ No results above threshold, lowering to 0.2")
    threshold = 0.2
    filtered_indices = [i for i in top_indices if scores[i] > threshold]

filtered_results = video_index.iloc[filtered_indices][['video_id', 'title']].copy()
filtered_results['score'] = scores[filtered_indices]

filtered_results

,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001


In [ ]:
# -----------------------------
# Search Function
# -----------------------------

def returnSearchResults(query, df, top_k=5, threshold=0.3):
    
    query_embedding = model.encode([query])
    
    scores = cosine_similarity(query_embedding, embeddings)[0]
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    filtered_indices = [i for i in top_indices if scores[i] > threshold]
    
    # fallback if empty
    if len(filtered_indices) == 0:
        threshold = 0.2
        filtered_indices = [i for i in top_indices if scores[i] > threshold]
    
    results = df.iloc[filtered_indices][['video_id', 'title']].copy()
    results['score'] = scores[filtered_indices]
    
    return results

In [ ]:
# -----------------------------
# Test Search
# -----------------------------

query = "What is overfitting?"

results = returnSearchResults(query, video_index)

results

,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001


In [ ]:
# -----------------------------
# Parameter Tuning
# -----------------------------

query = "What is overfitting?"

configs = [
    {"top_k": 3, "threshold": 0.3},
    {"top_k": 5, "threshold": 0.3},
    {"top_k": 5, "threshold": 0.2},
]

for config in configs:
    print(f"\nTop-K = {config['top_k']}, Threshold = {config['threshold']}")
    display(returnSearchResults(query, video_index, config['top_k'], config['threshold']))


Top-K = 3, Threshold = 0.3


,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001



Top-K = 5, Threshold = 0.3


,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001



Top-K = 5, Threshold = 0.2


,video_id,title,score
20,U6-RekkuORI,Closures in JavaScript explained with a simple...,0.310315
178,vK30T6wKauA,How to focus your efforts in the right directi...,0.300001
127,9kKNFP9srjc,So how do closures work again,0.261079
9,E7o_WdfKszU,Theres so much to learn so how do you focus to...,0.260392
30,oRZouQxC5Gw,When youre learning a new skill youve gotta be...,0.250224


In [ ]:
# -----------------------------
# Evaluation
# -----------------------------

queries_df = pd.read_csv("../data/query_video_mapping.csv")

evaluation_results = []

for _, row in queries_df.iterrows():
    query = row['query']
    true_video = row['relevant_video_id']
    
    results = returnSearchResults(query, video_index, top_k=10, threshold=0.2)
    
    if true_video in results['video_id'].values:
        rank = list(results['video_id']).index(true_video) + 1
    else:
        rank = None
    
    evaluation_results.append({
        "Query": query,
        "Expected": true_video,
        "Rank": rank
    })

eval_df = pd.DataFrame(evaluation_results)

eval_df.head()

,Query,Expected,Rank
0,what are programming fundamentals,MZVyASCY8sQ,1.0
1,how to start learning programming,q4Ovbh3tpU0,NaN
2,beginner coding concepts explained,U6-RekkuORI,NaN
3,basics of software development,vc5T3VmCar0,NaN
4,introduction to programming logic,Fi8vnYgMiHA,NaN


In [ ]:
# -----------------------------
#  Metrics
# -----------------------------

total = len(eval_df)

top1 = sum(eval_df['Rank'] == 1)
top3 = sum(eval_df['Rank'].apply(lambda x: x is not None and x <= 3))
top5 = sum(eval_df['Rank'].apply(lambda x: x is not None and x <= 5))

print("📊 Evaluation Metrics")
print("Top-1 Recall:", round(top1 / total, 2))
print("Top-3 Recall:", round(top3 / total, 2))
print("Top-5 Recall:", round(top5 / total, 2))

📊 Evaluation Metrics
Top-1 Recall: 0.19
Top-3 Recall: 0.38
Top-5 Recall: 0.44
